# Model 1: Random Forest Setup

- `breed_forest_full.csv`: individual AKC trait scores plus coat categories.
- `breed_forest_avg.csv`: AKC trait-group average scores plus coat categories.

`Average Rank` is not included in either modeling CSV because it is used to create `Popularity Tier`.

In [9]:
from pathlib import Path
import sys

for base_dir in [Path.cwd(), *Path.cwd().parents]:
    if (base_dir / "models").exists():
        sys.path.insert(0, str(base_dir))
        break

from models.setup_utils import (
    CATEGORICAL_TRAIT_COLS,
    build_avg_traits,
    data_dirs,
    find_project_root,
    load_interim_data,
    prepare_numeric_traits,
    add_popularity_tier,
    write_random_forest_csv,
)

PROJECT_ROOT = find_project_root()
INTERIM_DIR, PROCESSED_DIR = data_dirs(PROJECT_ROOT)

## Load Data

In [10]:
breed_traits, breed_ranks = load_interim_data(INTERIM_DIR)
breed_ranks = add_popularity_tier(breed_ranks)
breed_traits, numeric_trait_cols = prepare_numeric_traits(breed_traits)

## Full Trait CSV

In [11]:
breed_traits_full = breed_traits[["Breed", *numeric_trait_cols, *CATEGORICAL_TRAIT_COLS]].merge(
    breed_ranks[["Breed", "Popularity Tier"]],
    on="Breed",
    how="inner",
)

full_output_path, breed_traits_full_course = write_random_forest_csv(
    breed_traits_full,
    numeric_trait_cols,
    PROCESSED_DIR / "breed_forest_full.csv",
)

## Grouped Avg Trait CSV

In [12]:
breed_traits_avg, avg_trait_cols = build_avg_traits(breed_traits)
breed_traits_avg = breed_traits_avg.merge(
    breed_traits[["Breed", *CATEGORICAL_TRAIT_COLS]],
    on="Breed",
    how="inner",
).merge(
    breed_ranks[["Breed", "Popularity Tier"]],
    on="Breed",
    how="inner",
)

avg_output_path, breed_traits_avg_course = write_random_forest_csv(
    breed_traits_avg,
    avg_trait_cols,
    PROCESSED_DIR / "breed_forest_avg.csv",
)